In [0]:
print("Notebook is connected and running")

Notebook is connected and running


In [0]:
# Databricks notebook source
# Step 2 (Databricks/Azure): Load the 9 Olist CSVs into managed Delta tables.
#
# Run this as a Databricks notebook (paste into a notebook, or import as .py
# with "Databricks notebook source" format, which this file already has).

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
)

# ---- 1. CONFIG: update this to wherever your CSVs live ----
# Option A (Unity Catalog Volume, recommended to start):
BASE_PATH = "/Volumes/tgs_talk_to_my_data/tgs_talk_to_data/raw"
# Option B (Azure Blob/ADLS, once configured):
# BASE_PATH = "abfss://olist-raw@<storage-account>.dfs.core.windows.net"

SCHEMA_NAME = "tgs_talk_to_data"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")

# ---- 2. Explicit schemas (matches the types documented in Step 1) ----

orders_schema = StructType([
    StructField("order_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("order_status", StringType()),
    StructField("order_purchase_timestamp", TimestampType()),
    StructField("order_approved_at", TimestampType()),
    StructField("order_delivered_carrier_date", TimestampType()),
    StructField("order_delivered_customer_date", TimestampType()),
    StructField("order_estimated_delivery_date", TimestampType()),
])

customers_schema = StructType([
    StructField("customer_id", StringType()),
    StructField("customer_unique_id", StringType()),
    StructField("customer_zip_code_prefix", StringType()),
    StructField("customer_city", StringType()),
    StructField("customer_state", StringType()),
])

order_items_schema = StructType([
    StructField("order_id", StringType()),
    StructField("order_item_id", IntegerType()),
    StructField("product_id", StringType()),
    StructField("seller_id", StringType()),
    StructField("shipping_limit_date", TimestampType()),
    StructField("price", DoubleType()),
    StructField("freight_value", DoubleType()),
])

products_schema = StructType([
    StructField("product_id", StringType()),
    StructField("product_category_name", StringType()),
    StructField("product_name_lenght", IntegerType()),
    StructField("product_description_lenght", IntegerType()),
    StructField("product_photos_qty", IntegerType()),
    StructField("product_weight_g", IntegerType()),
    StructField("product_length_cm", IntegerType()),
    StructField("product_height_cm", IntegerType()),
    StructField("product_width_cm", IntegerType()),
])

category_translation_schema = StructType([
    StructField("product_category_name", StringType()),
    StructField("product_category_name_english", StringType()),
])

order_payments_schema = StructType([
    StructField("order_id", StringType()),
    StructField("payment_sequential", IntegerType()),
    StructField("payment_type", StringType()),
    StructField("payment_installments", IntegerType()),
    StructField("payment_value", DoubleType()),
])

order_reviews_schema = StructType([
    StructField("review_id", StringType()),
    StructField("order_id", StringType()),
    StructField("review_score", IntegerType()),
    StructField("review_comment_title", StringType()),
    StructField("review_comment_message", StringType()),
    StructField("review_creation_date", TimestampType()),
    StructField("review_answer_timestamp", TimestampType()),
])

sellers_schema = StructType([
    StructField("seller_id", StringType()),
    StructField("seller_zip_code_prefix", StringType()),
    StructField("seller_city", StringType()),
    StructField("seller_state", StringType()),
])

geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix", StringType()),
    StructField("geolocation_lat", DoubleType()),
    StructField("geolocation_lng", DoubleType()),
    StructField("geolocation_city", StringType()),
    StructField("geolocation_state", StringType()),
])

# ---- 3. File -> table -> schema mapping ----
tables = [
    ("olist_orders_dataset.csv", "orders", orders_schema),
    ("olist_customers_dataset.csv", "customers", customers_schema),
    ("olist_order_items_dataset.csv", "order_items", order_items_schema),
    ("olist_products_dataset.csv", "products", products_schema),
    ("product_category_name_translation.csv", "category_translation", category_translation_schema),
    ("olist_order_payments_dataset.csv", "order_payments", order_payments_schema),
    ("olist_order_reviews_dataset.csv", "order_reviews", order_reviews_schema),
    ("olist_sellers_dataset.csv", "sellers", sellers_schema),
    ("olist_geolocation_dataset.csv", "geolocation", geolocation_schema),
]

# ---- 4. Read each CSV and write as a managed Delta table ----
for filename, table_name, schema in tables:
    df = (
        spark.read
        .option("header", "true")
        .schema(schema)
        .csv(f"{BASE_PATH}/{filename}")
    )
    full_table_name = f"{SCHEMA_NAME}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_table_name)
    print(f"Loaded {full_table_name}: {df.count()} rows")

print("Done. Run: SHOW TABLES IN tgs_talk_to_data;")


Loaded tgs_talk_to_data.orders: 99441 rows
Loaded tgs_talk_to_data.customers: 99441 rows
Loaded tgs_talk_to_data.order_items: 112650 rows
Loaded tgs_talk_to_data.products: 32951 rows
Loaded tgs_talk_to_data.category_translation: 71 rows
Loaded tgs_talk_to_data.order_payments: 103886 rows
Loaded tgs_talk_to_data.order_reviews: 104162 rows
Loaded tgs_talk_to_data.sellers: 3095 rows
Loaded tgs_talk_to_data.geolocation: 1000163 rows
Done. Run: SHOW TABLES IN tgs_talk_to_data;


In [0]:
display(spark.sql("SHOW TABLES IN tgs_talk_to_data"))

database,tableName,isTemporary
tgs_talk_to_data,category_translation,false
tgs_talk_to_data,customers,false
tgs_talk_to_data,geolocation,false
tgs_talk_to_data,order_items,false
tgs_talk_to_data,order_payments,false
tgs_talk_to_data,order_reviews,false
tgs_talk_to_data,orders,false
tgs_talk_to_data,products,false
tgs_talk_to_data,sellers,false


In [0]:
display(spark.sql("""
SELECT
  date_trunc('MONTH', o.order_purchase_timestamp) AS month,
  SUM(oi.price) AS revenue
FROM tgs_talk_to_data.order_items oi
JOIN tgs_talk_to_data.orders o
  ON oi.order_id = o.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY month
ORDER BY month
"""))

month,revenue
2016-09-01T00:00:00.000Z,207.86
2016-10-01T00:00:00.000Z,44507.30000000016
2016-12-01T00:00:00.000Z,10.9
2017-01-01T00:00:00.000Z,120098.26999999952
2017-02-01T00:00:00.000Z,244959.3499999955
2017-03-01T00:00:00.000Z,368341.3200000005
2017-04-01T00:00:00.000Z,353842.98000000126
2017-05-01T00:00:00.000Z,503159.19000000984
2017-06-01T00:00:00.000Z,429916.61000000505
2017-07-01T00:00:00.000Z,492287.3000000101
